In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

In [ ]:
#salva os dados em um arquivo csv - esse sera o padrao que vamos manter

import json
import pandas as pd
caminho_arquivo = "../results/evaluation_rmse_mae_2.json"
arquivo_saida = "../results/evaluation_rmse_mae.csv"

with open(caminho_arquivo, 'r') as arquivo:
    data = json.load(arquivo)
csv_data = []

for key, metrics in data.items():
    parts = key.split()
    imputacao = parts[0]
    tcp = parts[3]
    link = parts[6]
    modelo = parts[-1].replace(",", "")  
    csv_data.append({
        "imputacao": imputacao,
        "link": link,
        "tcp": tcp,
        "modelo": modelo,
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"], 
        "NRMSE":metrics["NRMSE"]
    })

df = pd.DataFrame(csv_data)

df.to_csv(arquivo_saida, index=False)

In [ ]:
def get_link_list(dir):
    linklist = []
    print(dir)
    for file in os.listdir(dir):
        link = file.split(' ')[4]
        linklist.append(link)
    return linklist

In [ ]:
links = get_link_list("../datasets/choosen-best-svd/")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
arquivo = "../results/evaluation_rmse_mae.csv"
df = pd.read_csv(arquivo)

# Apply filters: TCP as 'bbr', Model as 'GRU', and specified links
df_bbr_filtered = df[(df['tcp'] == 'bbr') & (df['modelo'] == 'GRU') & (df['link'].isin(links))]

# Create pivot table
pivot_df = df_bbr_filtered.pivot_table(index=['link', 'tcp', 'modelo'], columns='imputacao', values='RMSE')

# Plotting
if not pivot_df.empty:
    pivot_df.plot(kind='barh', figsize=(10, 15))
    plt.title('Comparação de RMSE', fontsize=14)
    plt.xlabel('Link e TCP', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Imputação', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No data available after filtering. Check the filter criteria.")


In [ ]:
#gerando acuracia das predições

#salvar em um arquivo json a acuracia das predicoes
import os
import pandas as pd
import json

def organizar_arquivos_com_acuracia(raiz):
    files_by_technique = {}

    for pasta_raiz, subpastas, arquivos in os.walk(raiz):
        for arquivo in arquivos:
            if arquivo.endswith('.csv'):
                caminho_arquivo = os.path.join(pasta_raiz, arquivo)
                partes = arquivo.split(" ")
                try:
                    imputacao = partes[2]  # Nome do método de imputação
                    model = partes[1]
                    tc = partes[5]
                    link = partes[8]

                    modelo_tcp_key = f'{tc} {model}'

                    # Carregar o arquivo CSV
                    df = pd.read_csv(caminho_arquivo)

                    # Calcular a acurácia do arquivo
                    accuracy = calculate_accuracy(df)

                    # Criar entrada no dicionário files_by_technique
                    if imputacao not in files_by_technique:
                        files_by_technique[imputacao] = []

                    files_by_technique[imputacao].append({
                        "accuracy": accuracy,
                        "MODELO": model,
                        "TCP": tc,
                        "LINK": link
                    })
                
                except IndexError:
                    # Caso de arquivos que não possuem o mesmo formato esperado
                    continue
                except pd.errors.EmptyDataError:
                    # Arquivo vazio
                    print(f"Arquivo: {arquivo} - O arquivo está vazio, subpasta: {pasta_raiz}")
                except Exception as e:
                    # Tratar outras exceções, se necessário
                    print(f"Arquivo: {arquivo}, subpasta: {pasta_raiz} - Erro: {str(e)}")

    return files_by_technique

# Função para calcular a acurácia (substitua pela sua lógica real)
def calculate_accuracy(df):
    def check_accuracy(row):
        prediction = row['Prediction']
        test = row['Test']

        if prediction < 200 and test < 200:
            return 'r'
        elif 200 <= prediction < 500 and 200 <= test < 500:
            return 'o'
        elif 500 <= prediction < 800 and 500 <= test < 800:
            return 'y'
        elif 800 <= prediction < 1000 and 800 <= test < 1000:
            return 'b'
        elif prediction >= 1000 and test >= 1000:
            return 'g'
        else:
            return 'mismatch'
    
    df['Class'] = df.apply(check_accuracy, axis=1)
    df['Acerto'] = df['Class'] != 'mismatch'

    # Calcular a quantidade de vezes que cada letra aparece
    counts = df['Class'].value_counts()

    # Garantir que todas as letras ('r', 'o', 'y', 'b', 'g') estejam presentes
    for letter in ['r', 'o', 'y', 'b', 'g']:
        if letter not in counts.index:
            counts[letter] = 0

    total_acertos = df['Acerto'].sum()
    total_predictions = len(df)
    acuracia_total = (total_acertos / total_predictions) * 100

    return acuracia_total

#aplicando a funcao nos arquivos de predição
raiz = '../graficos/predicoes/round_2/valores-predicao'
data = organizar_arquivos_com_acuracia(raiz)

# Salvando os dados em um arquivo JSON
with open('../graficos/predicoes/round_2/predicoes_acuracia.json', 'w') as f:
    json.dump(data, f, indent=4)

In [ ]:
#cria um dicionario separando por link, tcp e modelo, para plotagem do grafico de barras posteriormente 
import json

# Caminho do arquivo JSON
arquivo_json = "../graficos/predicoes/round_2/predicoes_acuracia.json"

# Carregar o arquivo JSON para um dicionário
with open(arquivo_json, 'r') as arquivo:
    data = json.load(arquivo)

# Dados de accuracy para os links PR-AM e PA-BA
methods = ['interpolacao-linear', 'knn', 'media-movel', 'mediana-movel']
models = ['BBR LSTM', 'BBR GRU', 'CUBIC LSTM', 'CUBIC GRU']

accuracy_values_PRAM = {method: {model: [] for model in models} for method in methods}
accuracy_values_PABA = {method: {model: [] for model in models} for method in methods}
accuracy_values_MGRS = {method: {model: [] for model in models} for method in methods}
# Iterar sobre os dados do JSON e adicionar as acurácias aos dicionários correspondentes
for group, values in data.items():
    for item in values:
        try:
            accuracy = item['accuracy']
            model = item['MODELO']
            tcp = item['TCP'].upper()  # Convertendo para minúsculas
            link = item['LINK']

            #print(f"Acurácia: {accuracy}, Modelo: {model}, TCP: {tcp}, Link: {link}")

            # Ajuste para formatar corretamente as chaves dos modelos
            model_key = f'{tcp} {model}'
            #print(model_key)
            try:

                if link == 'pr-am' and model_key in models:
                    accuracy_values_PRAM[group][model_key].append(accuracy)
                elif link == 'pa-ba' and model_key in models:
                    accuracy_values_PABA[group][model_key].append(accuracy)
                elif link == 'mg-rs' and model_key in models:
                    accuracy_values_MGRS[group][model_key].append(accuracy)
            except:
                pass
        except KeyError as e:
            print(f"Erro ao processar item no JSON: {e}")

#plotar a accuracy para cada link
# Largura das barras
width = 0.15

# Posição das barras
x = np.arange(len(models))
# Tradução dos métodos para inglês
method_labels = {
    'interpolacao-linear': 'Linear Interpolation',
    'knn': 'KNN',
    'media-movel': 'Moving Average',
    'mediana-movel': 'Moving Median'
}

# Função para adicionar valores nas barras
def autolabel(ax, rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom')

# Função para criar e salvar o gráfico para PR-AM
def create_and_save_PRAM_graph():
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = []
    colors = ['#4168E1', '#67CB57', '#FF6961', '#9467bd']  # Azul, Verde, Laranja, Roxo
    for i, method in enumerate(methods):
        try:
            means = [np.mean(accuracy_values_PRAM[method][model]) for model in models]
            bars.append(ax.bar(x + i*width - 1.5*width, means, width, label=method_labels[method], color=colors[i]))
        except KeyError:
            pass
    
    ax.set_ylabel('Accuracy (%)', fontsize=17)
    ax.set_xlabel('', fontsize=14)
    ax.set_title('Accuracy for Forecasting Process of PR-AM', fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=17)
    ax.legend()

    plt.ylim(0, 100)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig("../graficos/predicoes/round_2/PR-AM_accuracy.png")
    plt.close()

# Função para criar e salvar o gráfico para MG-RS
def create_and_save_MGRS_graph():
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = []
    colors = ['#4168E1', '#67CB57', '#FF6961', '#9467bd']  # Azul, Verde, Laranja, Roxo
    for i, method in enumerate(methods):
        try:
            means = [np.mean(accuracy_values_MGRS[method][model]) for model in models]
            bars.append(ax.bar(x + i*width - 1.5*width, means, width, label=method_labels[method], color=colors[i]))
        except KeyError:
            pass

    ax.set_ylabel('Accuracy (%)', fontsize=17)
    ax.set_xlabel('', fontsize=14)
    ax.set_title('Accuracy for Forecasting Process of MG-RS', fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=17)
    ax.legend()

    plt.ylim(0, 100)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig("../graficos/predicoes/round_2\MG-RS_accuracy.png")
    plt.close()

# Função para criar e salvar o gráfico para PA-BA
def create_and_save_PABA_graph():
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = []
    colors = ['#4168E1', '#67CB57', '#FF6961', '#9467bd']  # Azul, Verde, Laranja, Roxo
    for i, method in enumerate(methods):
        try:
            means = [np.mean(accuracy_values_PABA[method][model]) for model in models]
            bars.append(ax.bar(x + i*width - 1.5*width, means, width, label=method_labels[method], color=colors[i]))
        except KeyError:
            pass

    ax.set_ylabel('Accuracy (%)', fontsize=17)
    ax.set_xlabel('', fontsize=14)
    ax.set_title('Accuracy for Forecasting Process of PA-BA', fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=17)
    ax.legend()

    plt.ylim(0, 100)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig("../graficos/predicoes/round_2/PA-BA_accuracy.png")
    plt.close()

# Criar e salvar os gráficos
create_and_save_PRAM_graph()
create_and_save_PABA_graph()
create_and_save_MGRS_graph()